# 04a — Logistic Regression: Environmental Analysis

Models MRSA acquisition as a function of ward-level colonization pressure, in the
**environmental** cohort. This cohort is matched on age, sex, prior surgery, room LOS, and
antibiotic exposure history — so age/sex are **not** added as covariates here (matching
already balances them; see the check in `01_eda.ipynb`).

Primary predictor: `MRSA_cp`. Other pathogens' CP columns and the Elixhauser index are
included as covariates to see whether MRSA-specific colonization pressure holds up once
general ward "dirtiness" and comorbidity burden are accounted for.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed

env = load_processed("environmental_mrsa.csv")
env.shape

## Build design matrix

In [ ]:
cp_cols = [c for c in env.columns if c.endswith("_cp")]
elix_cols = [c for c in env.columns if c.startswith("elix_") and c != "elix_index_mortality"]

predictors = cp_cols + ["any_surgery", "elix_index_mortality"]
model_df = env[["group_binary"] + predictors].dropna()
print(model_df.shape)
model_df.describe()

## Multicollinearity check (VIF)

In [ ]:
X = sm.add_constant(model_df[predictors])
vif = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif

## Fit logistic regression

In [ ]:
X = sm.add_constant(model_df[predictors])
y = model_df["group_binary"]

logit_env = sm.Logit(y, X).fit()
logit_env.summary()

## Odds ratios with 95% CIs

In [ ]:
params = logit_env.params
conf = logit_env.conf_int()
conf.columns = ["ci_low", "ci_high"]
or_table = np.exp(pd.concat([params, conf], axis=1).rename(columns={0: "coef"}))
or_table.columns = ["OR", "ci_low", "ci_high"]
or_table.sort_values("OR", ascending=False)

## Interpretation

Focus on `MRSA_cp`'s odds ratio and CI: does ward-level MRSA colonization pressure predict
acquisition after controlling for general ward colonization pressure (other CP columns),
prior surgery, and comorbidity burden?

In [ ]:
import pickle
Path("../reports").mkdir(exist_ok=True)
with open("../reports/logit_environmental.pkl", "wb") as f:
    pickle.dump(logit_env, f)